# Week 3: Object Detection — Chest X-Ray Pneumonia

**Goal:** Train a YOLOv8n model to detect pneumonia in chest X-rays.

---
## Step 0: Upload your data zip

Run the cell below and upload the `colab_package.zip` (generated locally by `package_for_colab.py`).

In [ ]:
from google.colab import files
import zipfile
import os
import json
import shutil
from pathlib import Path
import random
import yaml

random.seed(42)

print("Please upload colab_package.zip")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f"Extracting {zip_name}...")
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('dataset')
print("Extracted! Contents:")
os.system('ls -lh dataset/')

## Step 1: Install dependencies

In [ ]:
!pip install -q ultralytics

## Step 2: Convert COCO → YOLO format & create dataset config

Converts COCO JSON annotations into per-image YOLO .txt files, splits train/val, and writes dataset.yaml.

In [ ]:
DATA_DIR = Path('dataset')
OUT_DIR = Path('yolo_data')

TRAIN_RATIO = 0.8

with open(DATA_DIR / 'annotations_coco.json') as f:
    coco = json.load(f)

cat_map = {cat['id']: idx for idx, cat in enumerate(coco['categories'])}
class_names = [cat['name'] for cat in sorted(coco['categories'], key=lambda c: c['id'])]
print(f'Classes: {class_names}')

image_info = {img['id']: img for img in coco['images']}
anns_by_img = {}
for ann in coco['annotations']:
    anns_by_img.setdefault(ann['image_id'], []).append(ann)

img_ids = list(anns_by_img.keys())
random.shuffle(img_ids)

split = int(len(img_ids) * TRAIN_RATIO)
train_ids = set(img_ids[:split])
val_ids = set(img_ids[split:])
print(f'Train: {len(train_ids)}, Val: {len(val_ids)}')

for split_name, split_ids in [('train', train_ids), ('val', val_ids)]:
    (OUT_DIR / 'images' / split_name).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / 'labels' / split_name).mkdir(parents=True, exist_ok=True)

    for img_id in split_ids:
        img = image_info[img_id]
        fname = img['file_name']
        w, h = img['width'], img['height']

        src = DATA_DIR / 'images' / fname
        dst = OUT_DIR / 'images' / split_name / fname
        shutil.copy2(src, dst)

        lines = []
        for ann in anns_by_img[img_id]:
            cls_idx = cat_map[ann['category_id']]
            bx, by, bw, bh = ann['bbox']
            xc = (bx + bw / 2) / w
            yc = (by + bh / 2) / h
            nw = bw / w
            nh = bh / h
            lines.append(f"{cls_idx} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")

        label_path = OUT_DIR / 'labels' / split_name / (Path(fname).stem + '.txt')
        label_path.write_text('\n'.join(lines))

# Write dataset config
dataset_config = {
    'path': str(OUT_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(class_names),
    'names': class_names,
}
with open('dataset.yaml', 'w') as f:
    yaml.dump(dataset_config, f)

# Save class_names for later cells
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

print('Conversion complete!')
print(f'YOLO data saved to {OUT_DIR}')
print(f'dataset.yaml and class_names.json created')

## Step 3: Train + Evaluate + Visualize

Trains YOLOv8n, evaluates on validation set, saves side-by-side prediction images. This single cell ensures no path issues.

In [ ]:
import jsonfrom pathlib import Pathfrom PIL import Image, ImageDraw, ImageFontfrom ultralytics import YOLO# --- Config ---MODEL_DIR = Path('trained_model')PRED_DIR = Path('predictions')PRED_DIR.mkdir(exist_ok=True)MODEL_DIR.mkdir(exist_ok=True)with open('class_names.json') as f:    class_names = json.load(f)#  [A] TRAINprint('=' * 50)print('TRAINING')print('=' * 50)model = YOLO('yolov8n.pt')model.train(    data='dataset.yaml',    epochs=100,    patience=20,    batch=16,    imgsz=224,    project=str(MODEL_DIR),    name='yolov8n_chest_xray',    exist_ok=True,    pretrained=True,    optimizer='Adam',    lr0=0.001,    augment=True,    hsv_h=0.015,    hsv_s=0.7,    hsv_v=0.4,    degrees=10.0,    translate=0.1,    scale=0.1,    shear=2.0,    fliplr=0.5,    mosaic=0.5,    mixup=0.1,    verbose=True,)# Verify model existsmodel_path = MODEL_DIR / 'yolov8n_chest_xray' / 'weights' / 'best.pt'if not model_path.exists():
    model_path = Path('runs/detect') / str(MODEL_DIR) / 'yolov8n_chest_xray' / 'weights' / 'best.pt'
if not model_path.exists():
    # Fallback: search for any best.pt
    candidates = list(Path('.').rglob('**/best.pt'))    if candidates:        model_path = candidates[0]        print(f'Model found at: {model_path}')    else:        raise FileNotFoundError('best.pt not found after training!')else:    print(f'Model saved to: {model_path}')# Save a clean copy of just the trained weightsfinal_model_path = MODEL_DIR / 'best.pt'import shutilshutil.copy2(model_path, final_model_path)print(f'Clean model copy: {final_model_path}')# [B] EVALUATEprint('\n' + '=' * 50)print('EVALUATION')print('=' * 50)model = YOLO(str(model_path))metrics = model.val(data='dataset.yaml', imgsz=224, plots=True)results_dict = {    'mAP50': float(metrics.box.map50),    'mAP50_95': float(metrics.box.map),    'precision': float(metrics.box.mp),    'recall': float(metrics.box.mr),}if hasattr(metrics.box, 'ap_class_index') and metrics.box.ap_class_index is not None:    results_dict['class_maps'] = {        metrics.names[i]: float(metrics.box.maps[idx])        for idx, i in enumerate(metrics.box.ap_class_index)        if idx < len(metrics.box.maps)    }print(f'  mAP@0.5:      {results_dict["mAP50"]:.4f}')print(f'  mAP@0.5:0.95: {results_dict["mAP50_95"]:.4f}')print(f'  Precision:    {results_dict["precision"]:.4f}')print(f'  Recall:       {results_dict["recall"]:.4f}')if 'class_maps' in results_dict:    for cls, m in results_dict['class_maps'].items():        print(f'    {cls} mAP@0.5: {m:.4f}')with open(MODEL_DIR / 'metrics.json', 'w') as f:    json.dump(results_dict, f, indent=2)print(f'Metrics saved to {MODEL_DIR / "metrics.json"}')# [C] VISUALIZEprint('\n' + '=' * 50)print('VISUALIZATION')print('=' * 50)def draw_boxes(pil_img, boxes, colors, names, prefix=''):    draw = ImageDraw.Draw(pil_img)    font = ImageFont.load_default()    for cls_id, x1, y1, x2, y2 in boxes:        color = colors.get(cls_id, (255, 255, 0))        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)        label = f'{prefix}{names[cls_id]}'        bbox = draw.textbbox((x1, y1), label, font=font)        draw.rectangle(bbox, fill=color)        draw.text((x1, y1), label, fill=(0, 0, 0), font=font)    return pil_imgdef read_yolo_label(path, w, h):    if not path.exists():        return []    boxes = []    for line in path.read_text().strip().split('\n'):        if not line:            continue        parts = line.split()        cls_id = int(parts[0])        xc, yc, bw, bh = map(float, parts[1:])        x1 = int((xc - bw / 2) * w)        y1 = int((yc - bh / 2) * h)        x2 = int((xc + bw / 2) * w)        y2 = int((yc + bh / 2) * h)        boxes.append((cls_id, x1, y1, x2, y2))    return boxesval_dir = OUT_DIR / 'images' / 'val'label_dir = OUT_DIR / 'labels' / 'val'COLORS = {0: (0, 255, 0), 1: (255, 0, 0)}img_paths = sorted(val_dir.iterdir())results = model(list(img_paths), imgsz=224, conf=0.25)for img_path, result in zip(img_paths, results):    pil = Image.open(img_path).convert('RGB')    w, h = pil.size    gt_boxes = read_yolo_label(label_dir / f'{img_path.stem}.txt', w, h)    pred_boxes = []    if result.boxes is not None:        for box in result.boxes:            cls_id = int(box.cls[0])            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())            pred_boxes.append((cls_id, x1, y1, x2, y2))    gt_img = draw_boxes(pil.copy(), gt_boxes, COLORS, class_names, 'GT: ')    pred_img = draw_boxes(pil.copy(), pred_boxes, COLORS, class_names)    composite = Image.new('RGB', (w * 2, h))    composite.paste(gt_img, (0, 0))    composite.paste(pred_img, (w, 0))    composite.save(PRED_DIR / f'pred_{img_path.stem}.png')print(f'Saved {len(img_paths)} prediction images to {PRED_DIR}/')print('\nAll done! Move to Step 4 to download.')

## Step 4: Download results

Packages trained model, metrics, predictions, and training curves into a single zip.

In [ ]:
import zipfile

with zipfile.ZipFile('week3_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in MODEL_DIR.rglob('*'):
        if f.is_file():
            zf.write(f, str(f.relative_to('.')))
    for f in PRED_DIR.rglob('*'):
        if f.is_file():
            zf.write(f, str(f.relative_to('.')))
    # Also include training curves from the run dir
    for f in MODEL_DIR.rglob('*'):
        if f.is_file() and f.suffix in ('.png', '.jpg', '.csv'):
            zf.write(f, f'results/{f.relative_to(MODEL_DIR)}')

print(f'Created week3_results.zip ({os.path.getsize("week3_results.zip") / 1024:.1f} KB)')

from google.colab import files
files.download('week3_results.zip')

## Done!

Once downloaded, unzip `week3_results.zip`. It contains:
- `trained_model/best.pt` — trained model weights
- `trained_model/metrics.json` — mAP, precision, recall
- `predictions/` — side-by-side ground truth vs prediction images
- `results/` — training curves and confusion matrix

Submit these along with a short explanation (see `week3_report.md`).